In [1]:
import numpy as np
from tqdm import tqdm
import polars as pl
from icecream import ic
import pandas as pd
from numba import njit, prange
import multiprocessing as mp

In [2]:
@njit
def angle_between(v1: np.ndarray, v2: np.ndarray) -> float:
    norm_v1 = np.sqrt(np.dot(v1, v1))
    norm_v2 = np.sqrt(np.dot(v2, v2))
    
    if norm_v1 == 0 or norm_v2 == 0:
        return 0.0  # Handle zero-length vectors gracefully
    
    cosine_angle = np.dot(v1, v2) / (norm_v1 * norm_v2)
    
    # Clamp the cosine value to avoid numerical errors
    if cosine_angle < -1.0:
        cosine_angle = -1.0
    elif cosine_angle > 1.0:
        cosine_angle = 1.0
    
    angle_rad = np.arccos(cosine_angle)
    return np.rad2deg(angle_rad) 

In [3]:
a1 = np.array([ 3.16599989, 0.        ])
a2 = np.array([-1.58300031,  2.74183612    ])
a3 = np.array([0.0,0.0])
g1 = a1
g2 = a2
g3 = a3

In [99]:
A = np.array([a1,a2])
G = np.array([g1,g2])

xlim = ylim = (-5,5)
angles = np.arange(21.0,22.0,0.01)
results = []

theta = np.deg2rad(27.80)

R = np.array(
    [[np.cos(theta), -np.sin(theta)], 
     [np.sin(theta), np.cos(theta)]],
    dtype=np.float64,
)

B = G @ R.T 


AlphaLattice_alpha = []
for x in range(*xlim):
    for y in range(*ylim):
        if [x,y]==[0,0]:
            continue
        AlphaLattice_alpha.append([x, y])


In [100]:
G 

array([[ 3.16599989,  0.        ],
       [-1.58300031,  2.74183612]])

In [101]:
G @ R.T  

array([[ 2.80058327,  1.47658005],
       [-2.67904769,  1.68708587]])

In [102]:
R @ g1 

array([2.80058327, 1.47658005])

In [103]:
R @ g2

array([-2.67904769,  1.68708587])

In [104]:

AlphaLattice_alpha = np.vstack(AlphaLattice_alpha).T

In [105]:
AlphaLattice_iota = A @ AlphaLattice_alpha


In [106]:
sol = AlphaLattice_beta = np.linalg.solve(B, AlphaLattice_iota)

In [107]:
sol[0][:2], sol[1][:2]

(array([-2.09097167, -2.55735831]), array([-6.75483808, -5.8702571 ]))

In [108]:
solrounded = np.round(sol)
solfrac = sol - solrounded
solfrac_abs = np.abs(solfrac)

In [109]:
solfrac[0][:2]

array([-0.09097167,  0.44264169])

In [110]:
matches = []
good_matches = []
tol = 1e-2
for j in range(AlphaLattice_alpha.shape[1]):
    f1, f2 = solfrac[:, j]
    f1_abs, f2_abs = solfrac_abs[:, j]
    if f1_abs < tol:
        if f2_abs < tol:
            #print(f1_abs, f2_abs)
            m1_true, m2_true = sol[:, j]
            m1, m2 = solrounded[:, j]
            #print("True",{m1_true}, {m2_true})
            #print("Rounded",{m1},{m2})
            n1, n2 = AlphaLattice_alpha[:, j]
            #print("N",{n1},{n2})
            v1 = n1 * a1 + n2 * a2
            v2 = m1 * R@g1 + m2 * R@g2
            delvec = np.linalg.norm(v1 - v2)
            #print("Differnece", {delvec})
            if delvec < 0.1:
                matches.append((n1,n2,delvec))
                
for i in range(len(matches)):
    n_1, n_2, v_d = matches[i][0], matches[i][1], matches[i][2]
    for j in range(len(matches)):
        n_1p, n_2p = matches[j][0], matches[j][1]
        A1 = n_1 * a1 + n_2 * a2
        A2 = n_1p * a1 + n_2p * a2
        angle_A1_A2 = angle_between(A1,A2)
        delta_angle = np.abs(angle_A1_A2 - 60)
        if delta_angle < 0.1:
            good_matches.append((n_1, n_2, n_1p, n_2p, v_d, delta_angle, A1, A2))
        
                

In [111]:
matches

[]

In [86]:
# Define the column names
columns = [ 'n1', 'n2', 'n1p', 'n2p','vec_del', 'delta_angle', 'A1','A2']
df = pd.DataFrame(good_matches, columns=columns)
len(df)

0

In [87]:
# Sort the DataFrame by 'degree' in ascending order and 'norm' in descending order, followed by 'vec_del' and 'delta_angle'
df_sorted = df.sort_values(by=[ 'vec_del', 'delta_angle','n1','n2'], ascending=[True, True, False, False])
# Group by 'degree' and take the first entry in each group
final_df = df_sorted.groupby('vec_del', as_index=False).first()

len(final_df)

0

In [70]:
final_df

,vec_del,n1,n2,n1p,n2p,delta_angle,A1,A2
0,0.000517,4,2,-7,7,3.537707e-09,"[7.401873780000001, -4.27347382]","[0.0, 29.914316740000004]"
1,0.001035,8,4,-7,7,3.537707e-09,"[14.803747560000001, -8.54694764]","[0.0, 29.914316740000004]"
2,0.001369,7,-7,-8,-4,3.537707e-09,"[0.0, -29.914316740000004]","[-14.803747560000001, 8.54694764]"


In [122]:
import numpy as np
from scipy.linalg import null_space
import time
import itertools

# --- 1. Setup ---

# Your exact input vectors
a1 = np.array([3.16599989, 0.  ])
a2 = np.array([-1.58300031,  2.74183612])
b1 = np.array([2.80058327, 1.47658005])
b2 = np.array([-2.67904769,  1.68708587])

# Define the solution constraints
VAR_MIN, VAR_MAX = -5, 5
ERROR_TOLERANCE = 0.1 # New condition: ||Av|| < 0.1

# Construct the matrix A for the equation Av = 0
A = np.array([
    [a1[0], a2[0], -b1[0], -b2[0]],
    [a1[1], a2[1], -b1[1], -b2[1]]
])

# --- 2. Find Null Space (The Guide for Our Search) ---

K = null_space(A)
k1 = K[:, 0]
k2 = K[:, 1]
print("Null space basis (will guide our search):")
print(K)

# --- 3. Determine Parameter Bounds for c1 and c2 ---
# This estimation is a heuristic, but sufficient for these ranges
c_bound_estimate = VAR_MAX / (np.abs(K).sum(axis=1).min() + 1e-9)
c1_range = np.array([-c_bound_estimate, c_bound_estimate])
c2_range = np.array([-c_bound_estimate, c_bound_estimate])
print(f"\nEstimated parameter search range: c1, c2 in ~[{c1_range[0]:.0f}, {c1_range[1]:.0f}]")

# --- 4. Generate and Test Integer Candidates in the Neighborhood ---

print(f"\nSearching for integer vectors where the error is < {ERROR_TOLERANCE}...")
start_time = time.time()

found_solutions = set()

param_step = 0.5 # A reasonable step to sample the plane
c1_grid = np.arange(np.floor(c1_range[0]), np.ceil(c1_range[1]), param_step)
c2_grid = np.arange(np.floor(c2_range[0]), np.ceil(c2_range[1]), param_step)

# Generate the 16 offsets for the 4D hypercube corners
offsets = list(itertools.product([0, 1], repeat=4))

for c1 in c1_grid:
    for c2 in c2_grid:
        # 1. Get a point on the ideal, zero-error plane
        v_float = c1*k1 + c2*k2
        
        # 2. Get the base integer corner of the hypercube
        floor_v = np.floor(v_float).astype(int)

        # 3. Iterate through all 16 neighboring integer points
        for offset in offsets:
            v_int_candidate = floor_v + offset
            v_int_tuple = tuple(v_int_candidate)

            # 4. Check if we've already tested this integer candidate
            if v_int_tuple in found_solutions:
                continue

            # 5. Perform the two checks on this new candidate
            if np.all((v_int_candidate >= VAR_MIN) & (v_int_candidate <= VAR_MAX)):
                residual_vector = A @ v_int_candidate
                error_norm = np.linalg.norm(residual_vector)

                if error_norm < ERROR_TOLERANCE:
                    found_solutions.add(v_int_tuple)


print(f"\nFound {len(found_solutions)} solutions in {time.time() - start_time:.4f} seconds.")

if len(found_solutions) > 0:
    non_trivial_solutions = {s for s in found_solutions if s != (0, 0, 0, 0)}
    print(f"Found {len(non_trivial_solutions)} non-trivial solutions.")
    
    # Sort the solutions for consistent output
    sorted_solutions = sorted(list(non_trivial_solutions))
    
    print("\nSolutions (n1, n2, m1, m2):")
    for sol in sorted_solutions:
        print(sol)

Null space basis (will guide our search):
[[ 0.66981442 -0.40297925]
 [ 0.39251167  0.4846555 ]
 [ 0.62352718  0.01304675]
 [ 0.09217959  0.77623872]]

Estimated parameter search range: c1, c2 in ~[-8, 8]

Searching for integer vectors where the error is < 0.1...

Found 9 solutions in 0.1134 seconds.
Found 8 non-trivial solutions.

Solutions (n1, n2, m1, m2):
(np.int64(-5), np.int64(2), np.int64(-2), np.int64(5))
(np.int64(-4), np.int64(-1), np.int64(-3), np.int64(1))
(np.int64(-3), np.int64(-4), np.int64(-4), np.int64(-3))
(np.int64(-1), np.int64(3), np.int64(1), np.int64(4))
(np.int64(1), np.int64(-3), np.int64(-1), np.int64(-4))
(np.int64(3), np.int64(4), np.int64(4), np.int64(3))
(np.int64(4), np.int64(1), np.int64(3), np.int64(-1))
(np.int64(5), np.int64(-2), np.int64(2), np.int64(-5))


In [4]:
import numpy as np
import numba
from scipy.linalg import null_space
import multiprocessing
import time
from functools import partial

# A pre-generated NumPy array for the 16 hypercube corner offsets
offsets_np = np.array([
    [0, 0, 0, 0], [0, 0, 0, 1], [0, 0, 1, 0], [0, 0, 1, 1],
    [0, 1, 0, 0], [0, 1, 0, 1], [0, 1, 1, 0], [0, 1, 1, 1],
    [1, 0, 0, 0], [1, 0, 0, 1], [1, 0, 1, 0], [1, 0, 1, 1],
    [1, 1, 0, 0], [1, 1, 0, 1], [1, 1, 1, 0], [1, 1, 1, 1]
], dtype=np.int64)

@numba.jit(nopython=True, cache=True)
def find_solutions_jit(A, K, c1_range, c2_range, VAR_MIN, VAR_MAX, ERROR_TOLERANCE, offsets):
    """
    A Numba-optimized function to find solutions for a single system.
    This function contains the performance-critical loops.
    """
    max_solutions = 2000
    found_solutions = np.zeros((max_solutions, 4), dtype=np.int64)
    solution_count = 0
    
    found_hashes = np.zeros(max_solutions, dtype=np.uint64)

    k1 = K[:, 0]
    k2 = K[:, 1]
    param_step = 0.5

    for c1 in np.arange(c1_range[0], c1_range[1], param_step):
        for c2 in np.arange(c2_range[0], c2_range[1], param_step):
            v_float = c1 * k1 + c2 * k2
            floor_v = np.floor(v_float).astype(np.int64)

            for i in range(offsets.shape[0]):
                offset = offsets[i]
                v_int_candidate = floor_v + offset
                
                h = (1009 * v_int_candidate[0] + 1013 * v_int_candidate[1] + 
                     1021 * v_int_candidate[2] + 1031 * v_int_candidate[3])
                
                is_duplicate = False
                for j in range(solution_count):
                    if found_hashes[j] == h:
                        is_duplicate = True
                        break
                if is_duplicate:
                    continue

                in_range = True
                for j in range(v_int_candidate.shape[0]):
                    if not (v_int_candidate[j] >= VAR_MIN and v_int_candidate[j] <= VAR_MAX):
                        in_range = False
                        break
                if not in_range:
                    continue

                # --- THIS IS THE CORRECTED LINE ---
                # Explicitly cast the integer vector to float64 for the matmul
                residual_vector = A @ v_int_candidate.astype(np.float64)
                error_norm = np.linalg.norm(residual_vector)

                if error_norm < ERROR_TOLERANCE:
                    if solution_count < max_solutions:
                        found_solutions[solution_count] = v_int_candidate
                        found_hashes[solution_count] = h
                        solution_count += 1

    return found_solutions[:solution_count]

def process_single_b_pair(b1, b2, a1, a2, VAR_MIN, VAR_MAX, ERROR_TOLERANCE):
    """
    Worker function for one (b1, b2) pair.
    This will be executed in a separate process.
    """
    A = np.array([
        [a1[0], a2[0], -b1[0], -b2[0]],
        [a1[1], a2[1], -b1[1], -b2[1]]
    ])
    
    try:
        K = null_space(A)
    except np.linalg.LinAlgError:
        return np.empty((0, 4), dtype=np.int64)

    c_bound_estimate = VAR_MAX / (np.abs(K).sum(axis=1).min() + 1e-9)
    c1_range = np.array([-c_bound_estimate, c_bound_estimate])
    c2_range = np.array([-c_bound_estimate, c_bound_estimate])

    solutions = find_solutions_jit(A, K, c1_range, c2_range, VAR_MIN, VAR_MAX, ERROR_TOLERANCE, offsets_np)
    
    return solutions

if __name__ == "__main__":
    a1 = np.array([3.16599989, 0.])
    a2 = np.array([-1.58300031,  2.74183612])

    VAR_MIN, VAR_MAX = -4, 4
    ERROR_TOLERANCE = 0.1

    print("Generating list of b1, b2 pairs to process...")
    num_b_pairs = 10
    b_pairs = []
    base_b1 = np.array([2.80058327, 1.47658005])
    base_b2 = np.array([-2.67904769,  1.68708587])
    for i in range(num_b_pairs):
        angle = np.deg2rad(i * 360 / num_b_pairs)
        c, s = np.cos(angle), np.sin(angle)
        rotation_matrix = np.array([[c, -s], [s, c]])
        rotated_b1 = rotation_matrix @ base_b1
        rotated_b2 = rotation_matrix @ base_b2
        b_pairs.append((rotated_b1, rotated_b2))

    num_cores = multiprocessing.cpu_count()
    print(f"Starting processing on {num_cores} cores...")
    
    start_time = time.time()

    task_func = partial(process_single_b_pair,
                        a1=a1,
                        a2=a2,
                        VAR_MIN=VAR_MIN,
                        VAR_MAX=VAR_MAX,
                        ERROR_TOLERANCE=ERROR_TOLERANCE)

    with multiprocessing.Pool(processes=num_cores) as pool:
        all_results = pool.starmap(task_func, b_pairs)

    end_time = time.time()
    print(f"\nCompleted processing {num_b_pairs} pairs in {end_time - start_time:.2f} seconds.")

    total_solutions = 0
    for i, solutions in enumerate(all_results):
        if solutions.shape[0] > 0:
            total_solutions += solutions.shape[0]
            # Optional: print some results
            if total_solutions < 20: # Just print the first few to avoid spam
                print(f"Found {solutions.shape[0]} solutions for b_pair #{i}.")

    print(f"\nFound a total of {total_solutions} raw solutions across all runs.")

Generating list of b1, b2 pairs to process...
Starting processing on 20 cores...

Completed processing 10 pairs in 3.23 seconds.

Found a total of 151 raw solutions across all runs.


In [5]:
all_results

[array([[-3, -4, -4, -3],
        [-3, -4, -4, -3],
        [-3, -4, -4, -3],
        [-3, -4, -4, -3],
        [-3, -4, -4, -3],
        [-4, -1, -3,  1],
        [-4, -1, -3,  1],
        [-3, -4, -4, -3],
        [-3, -4, -4, -3],
        [-3, -4, -4, -3],
        [-3, -4, -4, -3],
        [-3, -4, -4, -3],
        [-4, -1, -3,  1],
        [-4, -1, -3,  1],
        [-4, -1, -3,  1],
        [-4, -1, -3,  1],
        [-4, -1, -3,  1],
        [-3, -4, -4, -3],
        [-3, -4, -4, -3],
        [-3, -4, -4, -3],
        [-3, -4, -4, -3],
        [-4, -1, -3,  1],
        [-4, -1, -3,  1],
        [-4, -1, -3,  1],
        [-4, -1, -3,  1],
        [-4, -1, -3,  1],
        [-3, -4, -4, -3],
        [-4, -1, -3,  1],
        [-4, -1, -3,  1],
        [-4, -1, -3,  1],
        [-4, -1, -3,  1],
        [-4, -1, -3,  1],
        [-4, -1, -3,  1],
        [-4, -1, -3,  1],
        [-4, -1, -3,  1],
        [-4, -1, -3,  1],
        [-4, -1, -3,  1],
        [-4, -1, -3,  1],
        [-4,